# 08 — Pipeline de Ibagué (02 + 03) en Cali

Mismo procedimiento que el notebook 06 (Villavicencio y Montería): ejecuta **copias** de `02_clima_canal_endemico_ibague_3` y `03_modelos_ibague_1` cambiando solo el municipio. Los originales no se tocan.

| Municipio | COD_MUN | Clima (GEE, diario) |
|---|---|---|
| Cali | 76001 | `Datos/clima/GEE_Clima_cali` |

Cambios a cada copia: `COD_MUN`, `MUNICIPIO`, `CLIMA_DIR`, `POBLACION` (opcional) y, en el 03, `procesado/modelos_76001/` para no sobrescribir Ibagué. `drive.mount` no se repite.

Salidas: `Colab Notebooks/02_clima_canal_endemico_cali.ipynb`, `03_modelos_cali.ipynb`; `procesado/serie_nivel2_76001.csv`, `tabla_modelado_76001.csv`, `comparacion_modelos_76001.csv`, `comparacion_clima_76001.csv`, `modelos_76001/`.

Ojo al leer: el test de Cali tiene muy pocos inicios de brote y una proporción de exceso muy alta. Sirve para ver si el patrón de Ibagué y Villavicencio se repite, no para elegir modelo.

**Advertencia (23-nov-2026): el clima de Cali no es representativo.** Los CSV de `GEE_Clima_cali` (ERA5-Land promediado en el polígono) dan entre 6.000 y 8.000 mm/año, lluvia casi todos los días, sin temporada seca, y 19,6 °C. Las cifras oficiales de la Alcaldía (IGAC, DAP, CVC, IDEAM) son 1.850 mm/año y 24,2 °C. Probablemente el polígono cubre los Farallones y no la zona urbana. Los resultados de Cali, sobre todo la comparación con y sin clima, son provisionales hasta volver a extraer el clima.

Además, el test 2023–2024 tiene 24 meses, 21 en exceso y 2 inicios (2025 queda sin canal): alertar todos los meses ya da F1 0,93, así que no sirve para elegir modelo. La validación 2019–2022 (48 meses, 4 inicios) es más informativa.

In [1]:
# 1. Configuración
import os, re, sys, time, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nbclient', 'nbformat', 'ipykernel'], check=True)
import nbformat
from nbclient import NotebookClient
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, average_precision_score

NB = '/content/drive/MyDrive/Colab Notebooks'
BASE = '/content/drive/MyDrive/Datos'
RUTA = f'{BASE}/dengue/procesado'
ORIG02 = f'{NB}/02_clima_canal_endemico_ibague_3.ipynb'
ORIG03 = f'{NB}/03_modelos_ibague_1.ipynb'
PRED05 = f'{BASE}/dengue/comparacion_dl_profundo/mensual/predicciones_test.csv'
MUNIS = {'cali': ('76001', 'Cali', 'GEE_Clima_cali')}

for p in (ORIG02, ORIG03):
    assert os.path.exists(p), f'No encuentro {p}'
for slug, (cod, nombre, carpeta) in MUNIS.items():
    d = f'{BASE}/clima/{carpeta}'
    n = len([f for f in os.listdir(d) if f.endswith('.csv')]) if os.path.isdir(d) else 0
    print(f'{nombre}: {n} CSV de clima en {d}')
    assert n > 0, f'Falta el acceso directo {d}'

# columnas que exige el 02
import glob
arch = sorted(glob.glob(f"{BASE}/clima/GEE_Clima_cali/**/*.csv", recursive=True))
print([os.path.basename(a) for a in arch])
cols = [c.strip().lower() for c in pd.read_csv(arch[0], nrows=3).columns]
print('Columnas:', cols)
faltan = [c for c in ['temp_media_c', 'temp_min_c', 'temp_max_c', 'precipitacion_mm'] if c not in cols]
assert not faltan and any(c in cols for c in ('fecha', 'date', 'time')), f'Faltan columnas: {faltan}'

Mounted at /content/drive
Cali: 17 CSV de clima en /content/drive/MyDrive/Datos/clima/GEE_Clima_cali
['clima_diario_cali_2009.csv', 'clima_diario_cali_2010.csv', 'clima_diario_cali_2011.csv', 'clima_diario_cali_2012.csv', 'clima_diario_cali_2013.csv', 'clima_diario_cali_2014.csv', 'clima_diario_cali_2015.csv', 'clima_diario_cali_2016.csv', 'clima_diario_cali_2017.csv', 'clima_diario_cali_2018.csv', 'clima_diario_cali_2019.csv', 'clima_diario_cali_2020.csv', 'clima_diario_cali_2021.csv', 'clima_diario_cali_2022.csv', 'clima_diario_cali_2023.csv', 'clima_diario_cali_2024.csv', 'clima_diario_cali_2025.csv']
Columnas: ['fecha', 'temp_media_c', 'temp_min_c', 'temp_max_c', 'precipitacion_mm']


In [2]:
# 2. Copia parametrizada de cada notebook
PARCHE = ("try:\n"
          "    import google.colab.drive as _d\n"
          "    _d.mount = lambda *a, **k: print('Drive ya está montado (notebook 06)')\n"
          "except Exception as e:\n"
          "    print('sin google.colab:', e)")

def parametrizar(nb, cod, nombre, carpeta, slug):
    cambios = [
        ('COD_MUN', r"^(\s*COD_MUN\s*=\s*)'\d{5}'", rf"\g<1>'{cod}'"),
        ('MUNICIPIO', r"^(\s*MUNICIPIO\s*=\s*)'[^']*'", rf"\g<1>'{nombre}'"),
        ('CLIMA_DIR', r"^(\s*CLIMA_DIR\s*=\s*)f?'[^']*'", rf"\g<1>f'{{BASE}}/clima/{carpeta}'"),
        ('POBLACION', r"^(\s*POBLACION\s*=\s*)f?'[^']*'", rf"\g<1>f'{{BASE}}/poblacion_{slug}.csv'"),
        ('SAL', r"^(\s*SAL\s*=\s*)f'\{RUTA\}/modelos'", r"\g<1>f'{RUTA}/modelos_{COD_MUN}'"),
    ]
    hechos = {nombre_c: 0 for nombre_c, _, _ in cambios}
    for c in nb.cells:
        if c.cell_type != 'code':
            continue
        for nombre_c, pat, rep in cambios:
            c.source, n = re.subn(pat, rep, c.source, flags=re.M)
            hechos[nombre_c] += n
        c.source = c.source.replace("'Ibagué — canal endémico'", f"'{nombre} — canal endémico'")
    nb.cells.insert(0, nbformat.v4.new_code_cell(PARCHE))
    restos = [l.strip() for c in nb.cells if c.cell_type == 'code' for l in c.source.splitlines()
              if '73001' in l or 'GEE_Clima_Ibague' in l or 'poblacion_ibague' in l]
    return nb, hechos, restos

def ejecutar(orig, destino, cod, nombre, carpeta, slug, es_03=False):
    nb = nbformat.read(orig, as_version=4)
    nb, hechos, restos = parametrizar(nb, cod, nombre, carpeta, slug)
    print(f'  {os.path.basename(orig)} → cambios aplicados: {hechos}')
    if restos:
        raise RuntimeError(f'Quedan referencias a Ibagué: {restos[:5]}')
    if hechos['COD_MUN'] != 1:
        raise RuntimeError('No encontré la línea COD_MUN')
    if es_03 and hechos['SAL'] != 1:
        raise RuntimeError('No pude redirigir procesado/modelos/: se sobrescribirían los archivos de Ibagué')
    t0, estado = time.time(), 'ok'
    try:
        NotebookClient(nb, timeout=1800, kernel_name='python3',
                       resources={'metadata': {'path': '/content'}}).execute()
    except Exception as e:
        estado = f'ERROR {type(e).__name__}: {str(e)[-800:]}'
    nbformat.write(nb, destino)
    print(f'  {os.path.basename(destino)}: {estado} ({time.time() - t0:.0f} s)')
    return estado

In [3]:
# 3. Correr 02 y 03 para cada municipio
ESTADO = {}
for slug, (cod, nombre, carpeta) in MUNIS.items():
    print(f'\n===== {nombre} ({cod}) =====')
    e2, e3 = '—', '—'
    try:
        e2 = ejecutar(ORIG02, f'{NB}/02_clima_canal_endemico_{slug}.ipynb', cod, nombre, carpeta, slug)
        if e2 == 'ok':
            t = pd.read_csv(f'{RUTA}/tabla_modelado_{cod}.csv')
            n_split = t['split'].value_counts().reindex(['train', 'val', 'test'], fill_value=0).to_dict()
            print('  filas por partición:', n_split)
            if min(n_split.values()) == 0:
                e3 = f'no se corre: partición vacía {n_split} (el canal no tiene umbral para esos años)'
                print('  03:', e3)
            else:
                e3 = ejecutar(ORIG03, f'{NB}/03_modelos_{slug}.ipynb', cod, nombre, carpeta, slug, es_03=True)
    except Exception as e:
        print('  ', type(e).__name__, e)
        e2 = e2 if e2 != '—' else f'ERROR {e}'
    ESTADO[nombre] = {'02': e2, '03': e3}
print(json.dumps(ESTADO, indent=1, ensure_ascii=False))


===== Cali (76001) =====
  02_clima_canal_endemico_ibague_3.ipynb → cambios aplicados: {'COD_MUN': 1, 'MUNICIPIO': 1, 'CLIMA_DIR': 1, 'POBLACION': 1, 'SAL': 0}
  02_clima_canal_endemico_cali.ipynb: ok (19 s)
  filas por partición: {'train': 82, 'val': 48, 'test': 24}
  03_modelos_ibague_1.ipynb → cambios aplicados: {'COD_MUN': 1, 'MUNICIPIO': 0, 'CLIMA_DIR': 0, 'POBLACION': 0, 'SAL': 1}
  03_modelos_cali.ipynb: ok (66 s)
{
 "Cali": {
  "02": "ok",
  "03": "ok"
 }
}


In [4]:
# 4. Canal endémico y resultados de cada municipio (Ibagué como referencia)
def resumen(cod, nombre):
    p2 = f'{RUTA}/serie_nivel2_{cod}.csv'
    if not os.path.exists(p2):
        print(f'\n##### {nombre}: no existe {p2}'); return
    n2 = pd.read_csv(p2, parse_dates=['mes']).set_index('mes')
    ex = n2['exceso'].dropna().loc['2012':]
    print(f'\n##### {nombre} ({cod})')
    print('Casos por año:', n2['casos'].groupby(n2.index.year).sum().loc[2012:].astype(int).to_dict())
    print('Proporción de meses en exceso por año:', ex.groupby(ex.index.year).mean().round(2).to_dict())
    print(f'Meses en exceso 2012–2025: {ex.mean():.0%} | meses con lluvia (2010+): {n2.loc["2010":, "lluvia_mm"].notna().mean():.0%}'
          f' | temperatura media: {n2["temp_c"].mean():.1f} °C')
    for f in (f'comparacion_modelos_{cod}.csv', f'comparacion_clima_{cod}.csv'):
        p = f'{RUTA}/{f}'
        print(f'\n{f}:')
        print(pd.read_csv(p).to_string(index=False) if os.path.exists(p) else '  no existe (el 03 no terminó)')

for cod, nombre in [('73001', 'Ibagué'), ('50001', 'Villavicencio')] + [(c, n) for c, n, _ in MUNIS.values()]:
    resumen(cod, nombre)


##### Ibagué (73001)
Casos por año: {2012: 2326, 2013: 5335, 2014: 3047, 2015: 5082, 2016: 2200, 2017: 603, 2018: 600, 2019: 5197, 2020: 2698, 2021: 317, 2022: 343, 2023: 4147, 2024: 13327, 2025: 1386}
Proporción de meses en exceso por año: {2012: 0.42, 2013: 0.92, 2014: 0.75, 2015: 1.0, 2016: 0.33, 2017: 0.0, 2018: 0.0, 2019: 0.67, 2020: 0.25, 2021: 0.0, 2022: 0.0, 2023: 0.5, 2024: 1.0, 2025: 0.0}
Meses en exceso 2012–2025: 42% | meses con lluvia (2010+): 100% | temperatura media: 17.7 °C

comparacion_modelos_73001.csv:
           modelo conjunto  recall  precision   f1  pr_auc  recall_inicios  n_inicios
     Persistencia      val    0.82       0.82 0.82     NaN             0.0        2.0
     Persistencia     test    0.89       0.89 0.89     NaN             0.0        2.0
        Logística      val    0.82       0.90 0.86    0.93             0.0        2.0
        Logística     test    1.00       0.67 0.80    0.92             1.0        2.0
          XGBoost      val    1.00       0

In [5]:
# 5. Comparación con el 05 (modelos globales, 39 municipios, sin clima local, ventana creciente)
if not os.path.exists(PRED05):
    print('No está', PRED05, '→ correr primero el 05')
else:
    print('Predicciones del 05 modificadas:', time.ctime(os.path.getmtime(PRED05)))
    pr = pd.read_csv(PRED05, dtype={'mun': str}, parse_dates=['fecha_t'])
    modelos = [c[2:] for c in pr.columns if c.startswith('p|')]

    def met(g, m):
        y, p = g['y'].values, g[f'p|{m}'].values
        pred = (p >= g[f'u|{m}'].values).astype(int)
        ini = (y == 1) & (g['exc_o'].values == 0)
        return {'f1': f1_score(y, pred, zero_division=0),
                'pr_auc': average_precision_score(y, p) if 0 < y.sum() < len(y) else np.nan,
                'recall_inicios': pred[ini].mean() if ini.any() else np.nan,
                'n': len(y), 'excesos': int(y.sum()), 'inicios': int(ini.sum())}

    for cod, nombre in [('73001', 'Ibagué')] + [(c, n) for c, n, _ in MUNIS.values()]:
        g = pr[pr['mun'] == cod]
        if g.empty:
            print(f'\n{nombre} ({cod}) no está entre los municipios del 05'); continue
        for etiqueta, sub in [('2016–2025', g), ('2023–2025 (mismo periodo que el test del 03)', g[g['fecha_t'].dt.year >= 2023])]:
            print(f'\n{nombre} — 05 global, {etiqueta}')
            print(pd.DataFrame({m: met(sub, m) for m in modelos}).T.round(3).to_string())
        p2 = f'{RUTA}/serie_nivel2_{cod}.csv'
        if os.path.exists(p2):
            e02 = pd.read_csv(p2, parse_dates=['mes']).set_index('mes')['exceso'].rename('exceso_02')
            comun = g.set_index('fecha_t')[['y']].join(e02, how='inner').dropna()
            print(f'Etiqueta de exceso igual en 05 y 02: {(comun["y"] == comun["exceso_02"]).mean():.1%} de {len(comun)} meses')

Predicciones del 05 modificadas: Fri Nov 20 23:50:50 2026

Ibagué — 05 global, 2016–2025
                            f1  pr_auc  recall_inicios      n  excesos  inicios
Persistencia             0.812   0.703             0.0  118.0     31.0      5.0
Binomial negativa local  0.817   0.889             0.8  118.0     31.0      5.0
LightGBM global          0.829   0.932             0.6  118.0     31.0      5.0
GRU global (1 capa)      0.853   0.949             0.6  118.0     31.0      5.0
LSTM global (2 capas)    0.841   0.916             0.6  118.0     31.0      5.0
TCN global               0.824   0.916             0.4  118.0     31.0      5.0

Ibagué — 05 global, 2023–2025 (mismo periodo que el test del 03)
                            f1  pr_auc  recall_inicios     n  excesos  inicios
Persistencia             0.889   0.846             0.0  36.0     18.0      2.0
Binomial negativa local  0.944   0.994             1.0  36.0     18.0      2.0
LightGBM global          0.923   0.977          

In [6]:
# 6. Exportar resultados (zip sobre un archivo marcador en Mi unidad)
import zipfile, shutil, glob, io, contextlib
DEST = '/content/drive/MyDrive/export_cali.zip'
buf = io.StringIO()
with contextlib.redirect_stdout(buf):
    for cod, nombre in [('73001', 'Ibagué'), ('50001', 'Villavicencio'), ('76001', 'Cali')]:
        resumen(cod, nombre)
    print('\nESTADO', json.dumps(ESTADO, ensure_ascii=False))
archivos = [f'{NB}/02_clima_canal_endemico_cali.ipynb', f'{NB}/03_modelos_cali.ipynb'] + \
           [f'{RUTA}/{f}' for f in ('serie_nivel2_76001.csv', 'tabla_modelado_76001.csv',
                                    'comparacion_modelos_76001.csv', 'comparacion_clima_76001.csv')] + \
           [p for p in glob.glob(f'{RUTA}/modelos_76001/*') if os.path.getsize(p) < 5e6]
with zipfile.ZipFile('/content/export_cali.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    z.writestr('resumen.txt', buf.getvalue())
    for p in archivos:
        if os.path.exists(p):
            z.write(p, os.path.relpath(p, '/content/drive/MyDrive'))
            print('ok ', p)
        else:
            print('falta', p)
assert os.path.exists(DEST), 'No está el marcador export_cali.zip en Mi unidad'
shutil.copyfile('/content/export_cali.zip', DEST)
print('Exportado', os.path.getsize(DEST), 'bytes')

ok  /content/drive/MyDrive/Colab Notebooks/02_clima_canal_endemico_cali.ipynb
ok  /content/drive/MyDrive/Colab Notebooks/03_modelos_cali.ipynb
ok  /content/drive/MyDrive/Datos/dengue/procesado/serie_nivel2_76001.csv
ok  /content/drive/MyDrive/Datos/dengue/procesado/tabla_modelado_76001.csv
ok  /content/drive/MyDrive/Datos/dengue/procesado/comparacion_modelos_76001.csv
ok  /content/drive/MyDrive/Datos/dengue/procesado/comparacion_clima_76001.csv
ok  /content/drive/MyDrive/Datos/dengue/procesado/modelos_76001/logistica.pkl
ok  /content/drive/MyDrive/Datos/dengue/procesado/modelos_76001/xgboost.json
ok  /content/drive/MyDrive/Datos/dengue/procesado/modelos_76001/binomial_negativa.pkl
ok  /content/drive/MyDrive/Datos/dengue/procesado/modelos_76001/predicciones_test.csv
Exportado 343090 bytes
